In [1]:
import pandas as pd
df = pd.read_csv('/home/datahouse1/raojingxin/myprojects/enzyme/metabolism/main_cases/pred_results_score_new.csv')
substrates = df['substrate'].to_list()
ecs = [(1,14) for _ in range(len(substrates))]
# dict_sub2ec = dict(zip(substrates,ecs))
# ecs = [i.strip() for i in ecs]

# # 凡是没有EC的我们用1.14代替
# ecs = [i.replace('None','1.14') for i in ecs]
# ecs = [i.split(' ') for i in ecs]
# ecs_new = []
# for eclist in ecs:
#     eclist = [tuple([int(j) for j in i.split('.')[:2]]) for i in eclist]
#     eclist = list(set(eclist))
#     ecs_new.append(eclist)

# subs_new = []
# for i in range(len(substrates)):
#     ecs = ecs_new[i]
#     num_repeat = len(ecs)
#     subs = [substrates[i] for _ in range(num_repeat)]
#     subs_new = subs_new + subs

# ecs_new = [i for k in ecs_new for i in k]

df_new = pd.DataFrame(columns=['substrate','ec','score'])
df_new['substrate'] = substrates
df_new['ec'] = ecs
df_new['score'] = ['None' for _ in range(len(substrates))]
df_new


,substrate,ec,score
0,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)O,"(1, 14)",None
1,CC(=O)NC1=CC=C2C(=C1)C(C1=CC=CC=C1Cl)=NCC(=O)N2,"(1, 14)",None
2,CC(=O)NC1=CC=CC(N2C(=O)N(C3CC3)C(=O)C3=C(NC4=C...,"(1, 14)",None
3,CC(C)(C)C1=CC(C(C)(C)C)=C(NC(=O)C2=CNC3=CC=CC=...,"(1, 14)",None
4,CC(C)(C)C1=CC(NC(=O)NC2=CC=C(C3=CN4C(=N3)SC3=C...,"(1, 14)",None
...,...,...,...
118,OC(C1=CC=CC=C1)(C1=CC=CC=C1)C12CC[N+](CCOCC3=C...,"(1, 14)",None
119,OC[C@H]1O[C@@H](C2=CC=C(Cl)C(CC3=CC=C(OCCOC4CC...,"(1, 14)",None
120,OCC1=NC=C2N1C1=CC=C(Cl)C=C1C(C1=CC=CC=C1F)=NC2,"(1, 14)",None
121,OC1=CC=C(COCC[N+]23CCC(C(O)(C4=CC=CC=C4)C4=CC=...,"(1, 14)",None


In [2]:
from tqdm import tqdm

import json
from os.path import isfile

import torch
from torch_geometric.data import Data
from rdkit.Chem import Draw
import requests

from IPython.display import SVG

from gnn_som import createGnnSom, loadGnnSomState
from gnn_som.MolFromKcf import MolFromKcfFile

import os

from rdkit import Chem
from rdkit.Chem import AllChem

os.environ['CUDA_VISIBLE_DEVICES'] = "2"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open('data/config.json', 'r') as f:
    config = json.load(f)
config['features']['enzyme'] = [tuple(ec) for ec in config['features']['enzyme']] 

models = []
for i, params in enumerate(config['models']):
    model = createGnnSom(*config['models'][i])
    loadGnnSomState(model, torch.load('data/model%d.pt' % i, map_location=torch.device('cpu')))
    models.append(model)

# Convert SMILES to .mol file using RDKit 

atom_scores_list = []

for i in tqdm(range(len(df_new))):
    smi = df_new.iloc[i,0]
    enzyme = df_new.iloc[i,1]


    # Create RDKit mol object
    # molecule = 'CC1([C@@H]2[C@H]1[C@H](N(C2)C(=O)[C@H](C(C)(C)C)NC(=O)C(F)(F)F)C(=O)N[C@@H](C[C@@H]3CCNC3=O)C#N)C'
    print(smi)
    molecule = Chem.MolFromSmiles(smi)
    molecule = Chem.AddHs(molecule)

    if os.path.exists("molecule.mol"):
        os.remove("molecule.mol")

    # Compute 2D coordenates and write to .mol file
    AllChem.Compute2DCoords(molecule)
    print(Chem.MolToMolBlock(molecule), file=open("molecule.mol", 'a+'))

    if os.path.exists("molecule.kcf"):
        os.remove("molecule.kcf")

    # Convert .mol to .kcf (KEGG format used in the paper) 
    # https://iwatobipen.wordpress.com/2016/12/23/convert-chemical-file-format/
    # https://www.genome.jp/tools/gn_ca_tools_api.html
    !curl -F molfile=@molecule.mol http://rest.genome.jp/mol2kcf/ > molecule.kcf

    # Define mol
    mol = MolFromKcfFile('molecule.kcf')

        # enzyme = (2,4)

    print(enzyme)
    numFeatures = sum(len(feature) for feature in config['features'].values())
    x = torch.zeros((mol.GetNumAtoms(), numFeatures), dtype=torch.float32)
    for atom in mol.GetAtoms():
        x[atom.GetIdx(), config['features']['enzyme'].index(enzyme)] = 1
        offset = len(config['features']['enzyme'])
        x[atom.GetIdx(), offset + config['features']['element'].index(atom.GetSymbol())] = 1
        offset += len(config['features']['element'])
        x[atom.GetIdx(), offset + config['features']['kcfType'].index(atom.GetProp('kcfType'))] = 1

    edgeIndex = torch.zeros((2, mol.GetNumBonds() * 2), dtype=torch.int64)
    for bond in mol.GetBonds():
        i = bond.GetIdx()
        edgeIndex[0][i * 2] = bond.GetBeginAtomIdx()
        edgeIndex[1][i * 2] = bond.GetEndAtomIdx()
        edgeIndex[0][i * 2 + 1] = bond.GetEndAtomIdx()
        edgeIndex[1][i * 2 + 1] = bond.GetBeginAtomIdx()

    data = Data(x=x, edgeIndex=edgeIndex)

    # 确保模型和数据都在同一设备上运行
    data = data.to(device)  # 将数据移动到 GPU 或保持在 CPU
    y = None

    for model in models:
        model = model.to(device)  # 将每个模型移动到 GPU（如果尚未移动）
        newY = torch.sigmoid(model(data.x, data.edgeIndex))  # 数据已在 GPU 或 CPU 上
        y = newY if y is None else torch.add(y, newY)

    # 计算模型平均输出
    y = torch.div(y, len(models))
    print(y.shape)

    # import torch

    # # 假设 scores 是你的 torch 张量，大小是 [14, 1]
    # scores = y  # 示例：14个原子的反应分数，实际使用时替换为你的数据

    # # 创建一个空字典来存储原子序号和对应的分数
    # atom_scores = {}

    # # 将分数映射到原子序号
    # for idx, score in enumerate(scores):
    #     atom_scores[idx] = score.item()  # 使用 item() 将张量转换为标量

    # 打印结果
    # print(atom_scores)

    atom_scores_list.append(y)

    print(f'substrate: {smi}, ec: {enzyme}, score: {y}')

/tmp/ipykernel_47104/173021330.py:32: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loadGnnSomState(model, torch.load('data/model%d.pt' % i, map_location=torch.device('cpu')

CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)O
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  6222    0  1696  100  4526   3111   8304 --:--:-- --:--:-- --:--:-- 11395
(1, 14)


  1%|          | 1/123 [00:03<07:44,  3.80s/it]

torch.Size([23, 1])
substrate: CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)O, ec: (1, 14), score: tensor([[0.0040],
        [0.0117],
        [0.0108],
        [0.0428],
        [0.0981],
        [0.0337],
        [0.2098],
        [0.1850],
        [0.1239],
        [0.1591],
        [0.0860],
        [0.1223],
        [0.1843],
        [0.1439],
        [0.0095],
        [0.0662],
        [0.0832],
        [0.0482],
        [0.0315],
        [0.1749],
        [0.1739],
        [0.0262],
        [0.1480]], device='cuda:0', grad_fn=<DivBackward0>)
CC(=O)NC1=CC=C2C(=C1)C(C1=CC=CC=C1Cl)=NCC(=O)N2
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  5099    0  1722  100  3377   5079   9961 --:--:-- --:--:-- --:--:-- 15041
(1, 14)


  2%|▏         | 2/123 [00:04<04:22,  2.17s/it]

...

torch.Size([24, 1])
substrate: OCC1=NC=C2N1C1=CC=C(Cl)C=C1C(C1=CC=CC=C1F)=NC2, ec: (1, 14), score: tensor([[0.9624],
        [0.9831],
        [0.3073],
        [0.1678],
        [0.0348],
        [0.1267],
        [0.2230],
        [0.0170],
        [0.0405],
        [0.0459],
        [0.0057],
        [0.0448],
        [0.0431],
        [0.0087],
        [0.1007],
        [0.0708],
        [0.2106],
        [0.1359],
        [0.1746],
        [0.1039],
        [0.0905],
        [0.0431],
        [0.2630],
        [0.1094]], device='cuda:0', grad_fn=<DivBackward0>)
OC1=CC=C(COCC[N+]23CCC(C(O)(C4=CC=CC=C4)C4=CC=CC=C4)(CC2)CC3)C=C1
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  8375    0  2464  100  5911   4947  11869 --:--:-- --:--:-- --:--:-- 16817
(1, 14)


 99%|█████████▉| 122/123 [03:46<00:01,  1.65s/it]

torch.Size([33, 1])
substrate: OC1=CC=C(COCC[N+]23CCC(C(O)(C4=CC=CC=C4)C4=CC=CC=C4)(CC2)CC3)C=C1, ec: (1, 14), score: tensor([[0.8272],
        [0.5704],
        [0.2418],
        [0.0238],
        [0.0054],
        [0.5217],
        [0.7458],
        [0.5693],
        [0.0570],
        [0.4404],
        [0.3916],
        [0.0441],
        [0.0470],
        [0.8608],
        [0.8554],
        [0.0460],
        [0.2163],
        [0.1941],
        [0.0818],
        [0.2473],
        [0.1749],
        [0.0440],
        [0.1898],
        [0.1895],
        [0.1014],
        [0.1286],
        [0.1132],
        [0.1617],
        [0.3347],
        [0.1602],
        [0.2943],
        [0.0315],
        [0.2196]], device='cuda:0', grad_fn=<DivBackward0>)
C1=CN=C(N2CCNCC2)N=C1
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3222    0   937  100  2285   3012   7347 --:--:-- --:--:-- --

100%|██████████| 123/123 [03:47<00:00,  1.85s/it]

torch.Size([12, 1])
substrate: C1=CN=C(N2CCNCC2)N=C1, ec: (1, 14), score: tensor([[0.3050],
        [0.1364],
        [0.3661],
        [0.3512],
        [0.5289],
        [0.5266],
        [0.4213],
        [0.3062],
        [0.3807],
        [0.3642],
        [0.3995],
        [0.2873]], device='cuda:0', grad_fn=<DivBackward0>)


In [3]:
import torch
atom_scores_list_new = []
# 假设 scores 是你的 torch 张量，大小是 [14, 1]
for i in atom_scores_list:
    scores = i  # 示例：14个原子的反应分数，实际使用时替换为你的数据

    # 创建一个空字典来存储原子序号和对应的分数
    atom_scores = {}

    # 将分数映射到原子序号
    for idx, score in enumerate(scores):
        atom_scores[idx] = score.item()  # 使用 item() 将张量转换为标量

# 打印结果
    atom_scores_list_new.append(atom_scores)

In [4]:
df_new['score'] = atom_scores_list_new

In [5]:
df_new

,substrate,ec,score
0,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)O,"(1, 14)","{0: 0.00403987243771553, 1: 0.0116602797061204..."
1,CC(=O)NC1=CC=C2C(=C1)C(C1=CC=CC=C1Cl)=NCC(=O)N2,"(1, 14)","{0: 0.11337683349847794, 1: 0.4291064441204071..."
2,CC(=O)NC1=CC=CC(N2C(=O)N(C3CC3)C(=O)C3=C(NC4=C...,"(1, 14)","{0: 0.1749592274427414, 1: 0.48370441794395447..."
3,CC(C)(C)C1=CC(C(C)(C)C)=C(NC(=O)C2=CNC3=CC=CC=...,"(1, 14)","{0: 0.19561421871185303, 1: 0.083653524518013,..."
4,CC(C)(C)C1=CC(NC(=O)NC2=CC=C(C3=CN4C(=N3)SC3=C...,"(1, 14)","{0: 0.23840060830116272, 1: 0.2645083367824554..."
...,...,...,...
118,OC(C1=CC=CC=C1)(C1=CC=CC=C1)C12CC[N+](CCOCC3=C...,"(1, 14)","{0: 0.8853639960289001, 1: 0.9299560785293579,..."
119,OC[C@H]1O[C@@H](C2=CC=C(Cl)C(CC3=CC=C(OCCOC4CC...,"(1, 14)","{0: 0.2678848206996918, 1: 0.15740716457366943..."
120,OCC1=NC=C2N1C1=CC=C(Cl)C=C1C(C1=CC=CC=C1F)=NC2,"(1, 14)","{0: 0.9624060988426208, 1: 0.9830583930015564,..."
121,OC1=CC=C(COCC[N+]23CCC(C(O)(C4=CC=CC=C4)C4=CC=...,"(1, 14)","{0: 0.8271753191947937, 1: 0.5703520178794861,..."


In [6]:
df_new.to_pickle('gnnsom_results_drugbank.pickle')